## Set Up Environment

In [ ]:
!conda install nltk
nltk.download('stopwords')

In [ ]:
import pandas as pd
import nltk
from ntlk.corpus import stopwords
stop_words = set(stopwords.words('english'))
import string
import collections

## Initialise Data

In [ ]:
speeches = pd.read_csv("speeches.csv", sep='|', usecols=['date','contents'])
speeches.dropna(inplace=True)
speeches = speeches.groupby("date")['contents'].apply(lambda x: " ".join(x.astype(str))).reset_index()
speeches

In [ ]:
fx = pd.read_csv("fx.csv")
fx.columns = ["date", "time_period", "exchange_rate"]
fx = fx[["date","exchange_rate"]]
fx

### Merging Data

In [ ]:
df = pd.merge(fx, speeches, how='left')
df['date']=pd.to_datetime(df['date'])
df.set_index('date',inplace=True)
df

### Remove entries with outliers or mistakes

In [ ]:
df.plot(kind="line",xlabel="date",ylabel="EUR/USD Reference Exchange Rate")

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

## Handle Missing Observations

In [ ]:
df['exchange_rate'].ffill(inplace=True)
df.isna().sum()

### Calculate Exchange Rate Return

In [ ]:
df['return'] = df['exchange_rate'].diff()/df['exchange_rate']
df

#### Separate Good News and Bad News

In [ ]:
df['good_news'] = (df['return']>0.5/100).astype(int)
df['bad_news'] = (df['return'] <-0.5/100).astype(int)
df

## Remove the Entries with NA content

In [ ]:
df.dropna(inplace=True)
df

## Generate and Store Good Indicators and Bad Indicators

In [ ]:
def get_word_freq(contents, stop_words, num_words):
    freq=dict()
    for word in content.split():
        word=word.strip(string.punctuation+'-')
        word=word.lower()
        if word not in stop_words and len(word):
            if word in freq:
                freq[word]+=1
            else:
                freq[word]=1

    freq=dict(sorted(freq.items(), key=lambda item: -item[1]))
    return list(freq.keys())[:num_words]

### Good and Bad Contents

In [ ]:
good_news_contents = df.contents[df.good_news==1].str.cat(sep=' ')
bad_news_contents = df.contents[df.bad_news==1].str.cat(sep=' ')

### Good and Bad Indicators

In [ ]:
good_indicators = get_word_freq(good_news_contents, stop_words, num_words=20)
bad_indicators = get_word_freq(bad_news_contents, stop_words, num_words=20)

In [ ]:
good_indicators

In [ ]:
bad_indicators